# Revenue Prediction Model

In this notebook we build a machine learning model to predict
future revenue using historical sales data from the e-commerce dataset.

Steps:
1. Load dataset
2. Create monthly revenue dataset
3. Feature engineering
4. Train machine learning model
5. Evaluate model
6. Predict next month revenue

## Import Required Libraries

We import pandas for data processing, scikit-learn for machine learning,
and joblib for saving the trained model.

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

## Load Dataset

Load the e-commerce sales dataset into a pandas dataframe.

In [3]:
df = pd.read_csv("../data/superstore.csv",encoding='latin1')

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
0,32298,CA-2012-124891,31-07-2012,31-07-2012,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,...,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,2309.650,7,0.0,762.1845,933.57,Critical
1,26341,IN-2013-77878,05-02-2013,07-02-2013,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,...,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,0.1,-288.7650,923.63,Critical
2,25330,IN-2013-71249,17-10-2013,18-10-2013,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,...,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,0.1,919.9710,915.49,Medium
3,13524,ES-2013-1579342,28-01-2013,30-01-2013,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,...,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless",2892.510,5,0.1,-96.5400,910.16,Medium
4,47221,SG-2013-4320,05-11-2013,06-11-2013,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,...,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.960,8,0.0,311.5200,903.04,Critical


## Convert Date Column

Convert the order date column into datetime format
so that we can extract time-based features.

In [4]:
df['Order Date'] = pd.to_datetime(df['Order Date'])

C:\Users\Tirupati\AppData\Local\Temp\ipykernel_7644\3072535395.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Order Date'] = pd.to_datetime(df['Order Date'])


## Create Monthly Sales Dataset

Aggregate sales at the monthly level to analyze revenue trends.

In [5]:
monthly_sales = df.groupby(df['Order Date'].dt.to_period('M'))['Sales'].sum()

monthly_sales = monthly_sales.reset_index()

monthly_sales.head()

,Order Date,Sales
0,2011-01,98898.48886
1,2011-02,91152.15698
2,2011-03,145729.36736
3,2011-04,116915.76418
4,2011-05,146747.83610


## Feature Engineering

Extract year and month from the order date.
These features will help the model learn time patterns.

In [6]:
monthly_sales['Order Date'] = monthly_sales['Order Date'].astype(str)
monthly_sales['Order Date'] = pd.to_datetime(monthly_sales['Order Date'])

monthly_sales['Year'] = monthly_sales['Order Date'].dt.year
monthly_sales['Month'] = monthly_sales['Order Date'].dt.month

monthly_sales.head()

,Order Date,Sales,Year,Month
0,2011-01-01,98898.48886,2011,1
1,2011-02-01,91152.15698,2011,2
2,2011-03-01,145729.36736,2011,3
3,2011-04-01,116915.76418,2011,4
4,2011-05-01,146747.83610,2011,5


## Define Features and Target

Features:
- Year
- Month

Target:
- Sales

In [7]:
X = monthly_sales[['Year','Month']]
y = monthly_sales['Sales']

## Train Test Split

Split the dataset into training and testing sets
to evaluate the model performance.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Train Linear Regression Model

We use Linear Regression to model the relationship
between time features and revenue.

In [9]:
model = LinearRegression()

model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


## Model Prediction

Use the trained model to predict revenue for test data.

In [10]:
predictions = model.predict(X_test)

predictions[:5]

array([239730.30704062, 320654.84538788, 217955.0023406 , 385980.75948794,
       174404.39294055])

## Model Evaluation

Evaluate the model using MAE and R² score.

In [11]:
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("MAE:", mae)
print("R2 Score:", r2)

MAE: 36790.29847360026
R2 Score: 0.7875914042407619
